![Original climate-globe illustration introducing Global warming levels with real Atlas data; illustrative, not a data map](assets/atlas-splash.png)

# 12 · Global warming levels with real Atlas data

**Guiding question:** Does a 2 °C warming level refer to one calendar year for every model?

**Data:** Real upstream CMIP6 warming-level table, unmodified, pinned in data/manifest.json  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-15  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** tables, figures, and any explicitly named local exports; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=12-global-warming-levels.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

No. A warming-level view aligns model-specific windows at a common amount
       of global warming. The bundled upstream CMIP6 table contains actual 20-year
       windows for several scenarios. Its numeric coordinates below are window midpoints,
       **not forecasts of the first year the observed planet reaches a threshold**.

       We retain both `NA` and `9999` literally and exclude them from date arithmetic.
       An absent window is not year zero and is not evidence of zero warming. This
       chapter reports source tokens without guessing whether absence means non-attainment,
       unavailable data, or a workflow-specific exclusion. The official Atlas warming-level
       calculation uses a specified high-forcing pathway; cross-scenario plots here explore
       the auxiliary table, not a recreation of the official display.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "plotly==6.5.0"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Parse and inspect the real table

In [2]:
gwl = warming_levels()
display(gwl.groupby(["scenario", "status"]).size().unstack(fill_value=0))
display(gwl[gwl.status == "window"].head())
assert (gwl.loc[gwl.status == "window", "end"] - gwl.loc[gwl.status == "window", "start"]).eq(19).all()

status,9999,NA,window
scenario,,,
ssp119,88,24,8
ssp126,36,54,30
ssp245,32,32,56
ssp370,36,10,74
ssp585,16,9,95


,member,scenario,level,raw,start,end,midpoint,status
36,AWI_AWI-CM-1-1-MR_r1i1p1f1,ssp585,1.5,2010-2029,2010.0,2029.0,2019.5,window
37,AWI_AWI-CM-1-1-MR_r1i1p1f1,ssp585,2.0,2027-2046,2027.0,2046.0,2036.5,window
38,AWI_AWI-CM-1-1-MR_r1i1p1f1,ssp585,3.0,2050-2069,2050.0,2069.0,2059.5,window
39,AWI_AWI-CM-1-1-MR_r1i1p1f1,ssp585,4.0,2070-2089,2070.0,2089.0,2079.5,window
44,CAS_FGOALS-g3_r1i1p1f1,ssp126,1.5,2067-2086,2067.0,2086.0,2076.5,window


## Compare valid model-specific windows

In [3]:
selected = gwl[(gwl.scenario == "ssp585") & (gwl.status == "window")]
fig = go.Figure()
for level in sorted(selected.level.unique()):
    part = selected[selected.level == level]
    fig.add_trace(go.Box(y=part.midpoint, name=f"{level:g} °C", boxpoints="all", jitter=.3,
        text=part.member, hovertemplate="%{text}<br>window midpoint %{y}<extra></extra>"))
fig.update_layout(title="Real CMIP6 auxiliary windows · SSP5-8.5", xaxis_title="Global warming level above 1850–1900", yaxis_title="20-year window midpoint")
show(fig)
display(selected.groupby("level").agg(n=("member", "size"), first_midpoint=("midpoint", "min"), last_midpoint=("midpoint", "max")))

Figure({
    'data': [{'boxpoints': 'all',
              'hovertemplate': '%{text}<br>window midpoint %{y}<extra></extra>',
              'jitter': 0.3,
              'name': '1.5 °C',
              'text': array(['AWI_AWI-CM-1-1-MR_r1i1p1f1', 'CCCR-IITM_IITM-ESM_r1i1p1f1',
                             'CCCma_CanESM5_r1i1p1f1', 'CMCC_CMCC-CM2-SR5_r1i1p1f1',
                             'CMCC_CMCC-ESM2_r1i1p1f1', 'CNRM-CERFACS_CNRM-CM6-1-HR_r1i1p1f2',
                             'CNRM-CERFACS_CNRM-CM6-1_r1i1p1f2', 'CNRM-CERFACS_CNRM-ESM2-1_r1i1p1f2',
                             'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1',
                             'EC-Earth-Consortium_EC-Earth3-CC_r1i1p1f1', 'INM_INM-CM4-8_r1i1p1f1',
                             'INM_INM-CM5-0_r1i1p1f1', 'IPSL_IPSL-CM6A-LR_r1i1p1f1',
                             'KIOST_KIOST-ESM_r1i1p1f1', 'MIROC_MIROC-ES2L_r1i1p1f2',
                             'MIROC_MIROC6_r1i1p1f1', 'MOHC_HadGEM3-GC31-LL_r1i1p1f3',
                             'MOHC_HadGEM3-GC31-MM_r1i1p1f3', 'MOHC_UKESM1-0-LL_r1i1p1f2',
                             'MPI-M_MPI-ESM1-2-LR_r1i1p1f1', 'MRI_MRI-ESM2-0_r1i1p1f1',
                             'NCAR_CESM2-WACCM_r1i1p1f1', 'NCC_NorESM2-MM_r1i1p1f1',
                             'NIMS-KMA_KACE-1-0-G_r1i1p1f1', 'NOAA-GFDL_GFDL-ESM4_r1i1p1f1',
                             'NUIST_NESM3_r1i1p1f1'], dtype=object),
              'type': 'box',
              'y': {'bdata': ('AAAAAACOn0AAAAAAAM6fQAAAAAAAcp' ... 'AAep9AAAAAAADen0AAAAAAAJKfQA=='),
                    'dtype': 'f8'}},
             {'boxpoints': 'all',
              'hovertemplate': '%{text}<br>window midpoint %{y}<extra></extra>',
              'jitter': 0.3,
              'name': '2 °C',
              'text': array(['AWI_AWI-CM-1-1-MR_r1i1p1f1', 'CCCR-IITM_IITM-ESM_r1i1p1f1',
                             'CCCma_CanESM5_r1i1p1f1', 'CMCC_CMCC-CM2-SR5_r1i1p1f1',
                             'CMCC_CMCC-ESM2_r1i1p1f1', 'CNRM-CERFACS_CNRM-CM6-1-HR_r1i1p1f2',
                             'CNRM-CERFACS_CNRM-CM6-1_r1i1p1f2', 'CNRM-CERFACS_CNRM-ESM2-1_r1i1p1f2',
                             'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1',
                             'EC-Earth-Consortium_EC-Earth3-CC_r1i1p1f1', 'INM_INM-CM4-8_r1i1p1f1',
                             'INM_INM-CM5-0_r1i1p1f1', 'IPSL_IPSL-CM6A-LR_r1i1p1f1',
                             'KIOST_KIOST-ESM_r1i1p1f1', 'MIROC_MIROC-ES2L_r1i1p1f2',
                             'MIROC_MIROC6_r1i1p1f1', 'MOHC_HadGEM3-GC31-LL_r1i1p1f3',
                             'MOHC_HadGEM3-GC31-MM_r1i1p1f3', 'MOHC_UKESM1-0-LL_r1i1p1f2',
                             'MPI-M_MPI-ESM1-2-LR_r1i1p1f1', 'MRI_MRI-ESM2-0_r1i1p1f1',
                             'NCAR_CESM2-WACCM_r1i1p1f1', 'NCC_NorESM2-MM_r1i1p1f1',
                             'NIMS-KMA_KACE-1-0-G_r1i1p1f1', 'NOAA-GFDL_GFDL-ESM4_r1i1p1f1',
                             'NUIST_NESM3_r1i1p1f1'], dtype=object),
              'type': 'box',
              'y': {'bdata': ('AAAAAADSn0AAAAAAAAGgQAAAAAAAmp' ... 'AAnp9AAAAAAAAJoEAAAAAAAMafQA=='),
                    'dtype': 'f8'}},
             {'boxpoints': 'all',
              'hovertemplate': '%{text}<br>window midpoint %{y}<extra></extra>',
              'jitter': 0.3,
              'name': '3 °C',
              'text': array(['AWI_AWI-CM-1-1-MR_r1i1p1f1', 'CCCR-IITM_IITM-ESM_r1i1p1f1',
                             'CCCma_CanESM5_r1i1p1f1', 'CMCC_CMCC-CM2-SR5_r1i1p1f1',
                             'CMCC_CMCC-ESM2_r1i1p1f1', 'CNRM-CERFACS_CNRM-CM6-1-HR_r1i1p1f2',
                             'CNRM-CERFACS_CNRM-CM6-1_r1i1p1f2', 'CNRM-CERFACS_CNRM-ESM2-1_r1i1p1f2',
                             'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1',
                             'EC-Earth-Consortium_EC-Earth3-CC_r1i1p1f1', 'INM_INM-CM4-8_r1i1p1f1',
                             'INM_INM-CM5-0_r1i1p1f1', 'IPSL_IPSL-CM6A-LR_r1i1p1f1',
                             'KIOST_KIOST-ESM_r1i1p1f1', 'MIROC_MIR

,n,first_midpoint,last_midpoint
level,,,
1.5,26,2007.5,2040.5
2.0,26,2022.5,2054.5
3.0,26,2040.5,2076.5
4.0,17,2054.5,2083.5


## Try it and explain

Change the scenario to ssp126. Explain why different sample sizes matter when comparing boxes.

**Interpretation check:** Different levels/scenarios can have different valid members. Compare a common member set when appropriate and retain missing-status counts; boxes are not equal-probability forecasts.

## Transfer to the official tools

In the Atlas, switch between future periods and warming levels. Inspect the model windows and the chosen baseline.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)
- [Source table and authors](https://github.com/ecmwf-projects/c3s-atlas/blob/main/auxiliar/GWLs/CMIP6_WarmingLevels.csv)
- [GWL implementation](https://github.com/ecmwf-projects/c3s-atlas/blob/main/book/notebooks/GWLs.ipynb)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](credits.md) and [data guide](data-guide.md).

**Continue:** [Uncertainty and bias adjustment](13-uncertainty-and-bias.ipynb)